In [1]:

from pathlib import Path
import json
import joblib
import pandas as pd

PROJECT_ROOT = Path(r"D:\FraudShield")
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "train_merged.parquet"

required_files = [
    MODEL_DIR / "lightgbm_final_baseline.joblib",
    MODEL_DIR / "feature_columns.json",
    REPORT_DIR / "final_test_metrics.json",
    REPORT_DIR / "shap_global_importance.csv",
]

for file in required_files:
    print(f"{'FOUND' if file.exists() else 'MISSING'}: {file}")

if not DATA_FILE.exists():
    raise FileNotFoundError(DATA_FILE)

model = joblib.load(MODEL_DIR / "lightgbm_final_baseline.joblib")

with open(MODEL_DIR / "feature_columns.json", encoding="utf-8") as f:
    feature_columns = json.load(f)

with open(REPORT_DIR / "final_test_metrics.json", encoding="utf-8") as f:
    test_metrics = json.load(f)

print("\nModel loaded.")
print("Expected feature count:", len(feature_columns))
print("Saved test average precision:", test_metrics["average_precision"])

FOUND: D:\FraudShield\models\lightgbm_final_baseline.joblib
FOUND: D:\FraudShield\models\feature_columns.json
FOUND: D:\FraudShield\reports\final_test_metrics.json
FOUND: D:\FraudShield\reports\shap_global_importance.csv

Model loaded.
Expected feature count: 432
Saved test average precision: 0.5257885759907768


In [2]:

df = pd.read_parquet(DATA_FILE)

overview = pd.DataFrame({
    "Metric": [
        "Total transactions",
        "Fraudulent transactions",
        "Legitimate transactions",
        "Fraud rate (%)",
        "Identity match rate (%)",
    ],
    "Value": [
        len(df),
        int(df["isFraud"].sum()),
        int((df["isFraud"] == 0).sum()),
        round(df["isFraud"].mean() * 100, 3),
        24.424,
    ],
})

display(overview)

,Metric,Value
0,Total transactions,590540.000
1,Fraudulent transactions,20663.000
2,Legitimate transactions,569877.000
3,Fraud rate (%),3.499
4,Identity match rate (%),24.424


In [3]:

product_analytics = (
    df.groupby("ProductCD", dropna=False)
      .agg(
          transaction_count=("isFraud", "size"),
          fraud_count=("isFraud", "sum"),
          average_transaction_amount=("TransactionAmt", "mean"),
    )
    .reset_index()
)

product_analytics["fraud_rate_percent"] = (
    product_analytics["fraud_count"]
    / product_analytics["transaction_count"]
    * 100
)

display(product_analytics.sort_values("transaction_count", ascending=False))

product_analytics.to_csv(
    REPORT_DIR / "dashboard_product_analytics.csv",
    index=False,
)

print("Product analytics saved.")

,ProductCD,transaction_count,fraud_count,average_transaction_amount,fraud_rate_percent
4,W,439670,8969,153.158554,2.039939
0,C,68519,8008,42.872353,11.687269
2,R,37699,1426,168.306188,3.782594
1,H,33024,1574,73.170058,4.766231
3,S,11628,686,60.269487,5.899553


Product analytics saved.


In [4]:

shap_importance = pd.read_csv(
    REPORT_DIR / "shap_global_importance.csv"
)

top_features = shap_importance.head(15)
display(top_features)

top_features.to_csv(
    REPORT_DIR / "dashboard_top_features.csv",
    index=False,
)

print("Dashboard feature importance saved.")

,feature,mean_absolute_shap
0,id_30,1.425172
1,R_emaildomain,0.746627
2,id_31,0.611830
3,id_33,0.563904
4,P_emaildomain,0.525054
5,TransactionAmt,0.493984
6,C13,0.447428
7,TransactionDT,0.431312
8,C1,0.325818
9,card1,0.298545


Dashboard feature importance saved.
